# Geospatial Data Analysis Lab: Steel Plants Dataset


**Learning Objectives:**
- Perform exploratory data analysis (EDA) on geospatial datasets
- Visualize geospatial data using interactive maps with Plotly
- Merge granular exposure / population data (LitPop) with asset locations
- Aggregate data at the company level
- Integrate geospatial visualizations into a Streamlit dashboard

**Dataset:** Download the Global Energy Monitor [Global Iron and Steel Tracker](https://globalenergymonitor.org/projects/global-iron-steel-tracker) (formerly Global Steel Plant Tracker). Use the plant-level download from that page and place the file in the same folder as this notebook (or update the load path accordingly).

---


## Submission info

Work in **groups of up to 4**. Fill in every member before submitting.

| # | Full name | Student ID |
|---|-----------|------------|
| 1 | Adem Ben Njima  | B00818393 |
| 2 | Kenza Gueddas | B00822443 |
| 3 | Jawad Mallat  | B00820811 |
| 4 | Erdem Sanver | B00818169 |
| 5 | Deha Tanaci | B00822530 |

**Group / repo name:** `aidams-lab1-<surname1>-<surname2>-...`  
**Submitter (one person): Jawad Mallat**  
**Repo URL: https://github.com/Jawadmallat/aidams-lab1-bennjima-gueddas-mallat-sanver-tanaci.git**  
**Streamlit Cloud URL (bonus):**  

### What to submit
- This notebook (`lab_1.ipynb`) with all parts completed and cells run
- `app.py` (Part 6)
- Processed data exports used by the dashboard (e.g. CSV/Parquet), if applicable
- (Bonus) Deployed Streamlit Cloud app link, if completed

<span style="color: #FFD700; font-weight: bold">Send submission info to my email (1 email per group)</span> — include the GitHub repo URL and, if you did the bonus, the Streamlit Cloud link.


## Upload to GitHub

Follow this checklist (one repo per group):

1. Create a **private** repository (or use the course organization if provided).
2. Name it using the pattern above, e.g. `aidams-lab1-ali-ben-chen-diaz`.
3. Add your files (`lab_1.ipynb`, `app.py`, exports, and a short `README.md` with how to run the dashboard).
5. Commit and push:
   ```bash
   git init
   git add lab_1.ipynb app.py .gitignore README.md
   git commit -m "Complete AIDAMS Lab 1"
   git branch -M main
   git remote add origin <YOUR_REPO_URL>
   git push -u origin main
   ```
6. Invite the instructor (or open the assignment link) and paste the **repo URL** in the Submission info table above.

**Done when:** all 4 names are filled in, the notebook contains all outputs (no need for the instructor to re-run it), maps are visible, and `streamlit run app.py` works from the repo.

**Bonus (optional):** deploy `app.py` to Streamlit Cloud and paste the public app URL above / in your submission email.


## Part 1: Setup and Data Loading

Import the necessary libraries and load the steel plants dataset.

**Tip:** After loading, run `df.columns` and `df.head()`. Column names in the file may differ slightly by release — inspect them and adapt your code accordingly.


In [36]:
# Import required libraries
from __future__ import annotations

import glob
import re
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from sklearn.neighbors import BallTree

# --- Reproducibility -------------------------------------------------------
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# --- Display / plotting defaults ------------------------------------------
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pio.templates.default = "plotly_white"

# Renderer: embed BOTH the plotly mime bundle (JupyterLab / VS Code) and a
# self-contained HTML div loading plotly.js from a CDN, so the saved figures
# are also visible on GitHub / nbviewer without re-running the notebook.
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Open basemap style: renders without a Mapbox access token.
MAP_STYLE = "open-street-map"

# --- Paths (relative, so the notebook runs anywhere in the repo) ----------
BASE_DIR = Path.cwd()
GEM_DIR = BASE_DIR / "gem-data"
LITPOP_DIR = BASE_DIR / "litpop"
EXPORT_DIR = BASE_DIR / "data" / "processed"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

PLANT_XLSX = GEM_DIR / "Plant-level_data_Global_Iron_and_Steel_Tracker_June_2026_V1.xlsx"

import plotly
print(f"pandas {pd.__version__} | numpy {np.__version__} | plotly {plotly.__version__}")
print(f"Base dir : {BASE_DIR}")
print(f"Exports  : {EXPORT_DIR.relative_to(BASE_DIR)}")
assert PLANT_XLSX.exists(), f"Missing {PLANT_XLSX}"

pandas 3.0.5 | numpy 2.5.3 | plotly 7.1.0
Base dir : /Users/jawadmallat/Desktop/AIDAMS_3A/research_course/Lab1
Exports  : data/processed


In [37]:
# Load the steel plants dataset
#
# The GEM workbook splits information across sheets:
#   "Plant data"                  -> one row per plant (identity, owner, location, age)
#   "Plant capacities and status" -> one row per production UNIT (a plant can have several)
# Capacity therefore has to be aggregated up to the plant before it can be merged.

OPERATING_STATUSES = ("operating", "operating pre-retirement")
CAPACITY_COL = "Nominal crude steel capacity (ttpa)"

print("Sheets:", pd.ExcelFile(PLANT_XLSX).sheet_names, end="\n\n")

plants_raw = pd.read_excel(PLANT_XLSX, sheet_name="Plant data")
capacities_raw = pd.read_excel(PLANT_XLSX, sheet_name="Plant capacities and status")

print(f"'Plant data'                  : {plants_raw.shape[0]:,} rows x {plants_raw.shape[1]} cols")
print(f"'Plant capacities and status' : {capacities_raw.shape[0]:,} rows x {capacities_raw.shape[1]} cols")
print(f"  -> {capacities_raw['GEM plant ID'].nunique():,} distinct plants "
      f"({capacities_raw.shape[0] - capacities_raw['GEM plant ID'].nunique():,} extra unit rows)")

plants_raw.head()

Sheets: ['About', 'Metadata', 'Plant data', 'Plant capacities and status', 'Plant production']

'Plant data'                  : 1,293 rows x 44 cols
'Plant capacities and status' : 1,845 rows x 16 cols
  -> 1,293 distinct plants (552 extra unit rows)


,GEM plant ID,Plant name (English),Plant name (other language),Other plant names (English),Other plant names (other language),Owner,Owner (other language),Owner GEM entity ID,Owner PermID,SOE status,Parent (English),Parent GEM entity ID,Parent PermID,Location address,Location address (other language),Municipality,Subnational unit,Country/area,Region,Coordinates,Coordinate accuracy,GEM wiki page,Plant age,Announced date,Construction date,Start date,Pre-retirement announcement date,Idled date,Retired date,Ferronickel capacity (ttpa),Sinter plant capacity (ttpa),Coking plant capacity (ttpa),Pelletizing plant capacity (ttpa),Category steel product,Steel products,Steel sector end users,Workforce size,ISO 14001,ISO 50001,ResponsibleSteel certification,Main production equipment,Power source,Iron ore source,Met coal source
0,P100000120882,Aba Iron and Steel Payas plant,ABA DEMİR ÇELİK,"EEY Iron and Steel, Nursan Steel Payas Plant (...",NaN,ABA Çelik Demir LŞ,ABA Demir ve Çelik İthalat İhracat Ticaret,E100000131190,unknown,NaN,ABA Çelik Demir LŞ [100.0%],E100000131190 [100.0%],unknown [100.0%],"3 Sanayi Bolgesi Sincan, Payas, Hatay, 31900 T...",NaN,Payas,Hatay,Türkiye,Europe,"36.7474130, 36.2173300",exact,https://www.gem.wiki/Aba_Iron_and_Steel_Payas_...,43,unknown,unknown,1983,unknown,unknown,unknown,NaN,NaN,NaN,NaN,"crude, semi-finished, finished rolled","billet, wire rod, angle, flat, bar, square bar...",unknown,900,unknown,unknown,no,EAF,unknown,unknown,NaN
1,P100000120753,Abba Steel Ohangwena steel plant,NaN,Groot Suisse Oshana plant,NaN,Abba Steel Ltd,NaN,E100001012072,unknown,NaN,Abba Steel Ltd [100.0%],E100001012072 [100.0%],unknown [100.0%],"Eeshoke Village, Oshikango, Ohangwena Region, ...",NaN,Oshikango,Ohangwena,Namibia,Africa,"-17.3978660, 15.8910220",approximate,https://www.gem.wiki/Abba_Steel_Ohangwena_stee...,1,2019,2020,2025,unknown,unknown,unknown,NaN,unknown,NaN,unknown,"semi-finished, finished rolled","billet, wire rod, rebar",building and infrastructure,5500,unknown,unknown,no,EAF,unknown,unknown,NaN
2,P100000120802,Abinsk Electric Steel Works,АЭМЗ,"AESW, ASW, AEMZ, AEMK",Абинский ЭлектроМеталлургический завод,Abinski Elektrometallurgicheski Zavod LLC,"ООО ""АБИНСКИЙ ЭЛЕКТРОМЕТАЛЛУРГИЧЕСКИЙ ЗАВОД""",E100000130999,5039667129,NaN,Novostal-M LLC [100.0%],E100000131027 [100.0%],5046391773 [100.0%],"Promyshlennaya, 4, Abinsk, Krasnodarskiy Kray,...","353320, Российская Федерация, Краснодарский Кр...",Abinsk,Krasnodar,Russia,Eurasia,"44.8819380, 38.1275100",exact,https://www.gem.wiki/Abinsk_Electric_Steel_Works,12,unknown,2008,2014,unknown,unknown,unknown,NaN,NaN,NaN,NaN,"semi-finished, finished rolled","wire rod, rebar, bar, billet, round bar, wire",unknown,4500,2025-10-06 00:00:00,unknown,no,EAF,unknown,NaN,unknown
3,P100000120020,Abul Khair Steel Sitakunda plant,আবুল খায়ের স্টিল মেল্টিং লিমিটেড,"AKS Long Steel, AKS Sitakund, AKS Melting, Abu...",NaN,Abul Khair Steel Ltd,NaN,E100000131068,5074007077,NaN,Abul Khair Ltd,E100000131048,5055443373,"Shitalpur, Sitakund, Chittagong, Bangladesh",NaN,Sitakunda,Chittagong,Bangladesh,Asia Pacific,"22.4720080, 91.7348270",exact,https://www.gem.wiki/Abul_Khair_Steel_Sitakund...,11,unknown,unknown,2015,unknown,unknown,unknown,NaN,NaN,NaN,NaN,"semi-finished, finished rolled","billet, rebar","building and infrastructure, energy",unknown,unknown,unknown,no,EAF,"100MW Power Plant, in progress 50MW solar plant",unknown,unknown
4,P100000120620,Acciaierie d'Italia Taranto steel plant,NaN,"ILVA Taranto steel plant (predecessor), ILVA S...",NaN,Acciaierie d'italia SpA,NaN,E100001010116,5067495106,Full,Acciaierie d'italia SpA [100.0%],E100001010116 [100.0%],5067495106 [100.0%],"Via Appia SS km 648, 74123 Taranto TA, Italy",NaN,Taranto,Province of Taranto,Italy,Europe,"40.5089930, 17.2075890",exact,https://www.gem.wiki/Acciaierie_d'Italia_Taran...,61,unknown,unknown,1965,unknown,unknown,unknown,NaN,>0,>0,NaN,"semi-finished, finished rolled","pipe, tube, flat","automotive, building and infrastr

---
## Part 2: Exploratory Data Analysis

Answer the following questions through your analysis:


### Question 1: Data Overview
**Task:** Display basic information about the dataset.
- How many steel plants are in the dataset?
- What are the column names and data types? (use `df.columns` / `df.dtypes` or `df.info()`, then adapt later code to the names you see)
- Are there any missing values?


In [38]:
# Display dataset shape
n_rows, n_cols = plants_raw.shape
print(f"Steel plants in the tracker : {n_rows:,}")
print(f"Attributes per plant        : {n_cols}")
print(f"Countries / areas covered   : {plants_raw['Country/area'].nunique()}")
print(f"Regions covered             : {plants_raw['Region'].nunique()}")
print(f"Distinct owners             : {plants_raw['Owner'].nunique():,}")
print(f"Duplicate GEM plant IDs     : {plants_raw['GEM plant ID'].duplicated().sum()}")

display(Markdown(f"""
> ### 📊 Dataset overview
> | Metric | Value |
> |---|---|
> | **Steel plants tracked** | **{n_rows:,}** |
> | Attributes per plant | {n_cols} |
> | Countries / areas covered | {plants_raw['Country/area'].nunique()} |
> | Regions covered | {plants_raw['Region'].nunique()} |
> | Distinct owners | {plants_raw['Owner'].nunique():,} |
> | Duplicate GEM plant IDs | {plants_raw['GEM plant ID'].duplicated().sum()} |
""".strip()))

Steel plants in the tracker : 1,293
Attributes per plant        : 44
Countries / areas covered   : 91
Regions covered             : 7
Distinct owners             : 1,069
Duplicate GEM plant IDs     : 0


> ### 📊 Dataset overview
> | Metric | Value |
> |---|---|
> | **Steel plants tracked** | **1,293** |
> | Attributes per plant | 44 |
> | Countries / areas covered | 91 |
> | Regions covered | 7 |
> | Distinct owners | 1,069 |
> | Duplicate GEM plant IDs | 0 |

In [39]:
# Display column information and data types
print("Columns in 'Plant data':")
for i, col in enumerate(plants_raw.columns, start=1):
    print(f"  {i:2d}. {col}  ->  {plants_raw[col].dtype}")

print("\nColumns in 'Plant capacities and status':")
for i, col in enumerate(capacities_raw.columns, start=1):
    print(f"  {i:2d}. {col}  ->  {capacities_raw[col].dtype}")

plants_raw.info()

Columns in 'Plant data':
   1. GEM plant ID  ->  str
   2. Plant name (English)  ->  str
   3. Plant name (other language)  ->  str
   4. Other plant names (English)  ->  str
   5. Other plant names (other language)  ->  str
   6. Owner  ->  str
   7. Owner (other language)  ->  str
   8. Owner GEM entity ID  ->  str
   9. Owner PermID  ->  object
  10. SOE status  ->  str
  11. Parent (English)  ->  str
  12. Parent GEM entity ID  ->  str
  13. Parent PermID  ->  object
  14. Location address  ->  str
  15. Location address (other language)  ->  str
  16. Municipality  ->  str
  17. Subnational unit  ->  str
  18. Country/area  ->  str
  19. Region  ->  str
  20. Coordinates  ->  str
  21. Coordinate accuracy  ->  str
  22. GEM wiki page  ->  str
  23. Plant age  ->  object
  24. Announced date  ->  object
  25. Construction date  ->  object
  26. Start date  ->  object
  27. Pre-retirement announcement date  ->  object
  28. Idled date  ->  object
  29. Retired date  ->  object
  30.

In [40]:
# Check for missing values
missing = (
    pd.DataFrame({
        "n_missing": plants_raw.isna().sum(),
        "pct_missing": plants_raw.isna().mean() * 100,
    })
    .query("n_missing > 0")
    .sort_values("n_missing", ascending=False)
)

print(f"{len(missing)} of {plants_raw.shape[1]} columns contain missing values.\n")
display(missing)

# Columns this analysis depends on -- these must be usable.
key_cols = ["GEM plant ID", "Plant name (English)", "Owner",
            "Country/area", "Region", "Coordinates", "Plant age"]
print("Completeness of the columns used downstream:")
for col in key_cols:
    n_null = plants_raw[col].isna().sum()
    print(f"  {col:<24} missing: {n_null:>4}  ({n_null / len(plants_raw):.1%})")

# NOTE: dtypes are 'object' for some numeric-looking fields because the source
# encodes unknowns as text. Confirmed sentinel values:
print("\nNon-numeric sentinels found in numeric-looking columns:")
print("  'Plant age' :", sorted({v for v in plants_raw["Plant age"].dropna().unique()
                                 if isinstance(v, str)}))
print(f"  '{CAPACITY_COL}' :",
      sorted({v for v in capacities_raw[CAPACITY_COL].dropna().unique() if isinstance(v, str)}))

_key_missing = ", ".join(
    f"**{col}** ({plants_raw[col].isna().sum()})"
    for col in key_cols if plants_raw[col].isna().sum() > 0
) or "none"
display(Markdown(f"""
> ### 🧹 Missing-data summary
> - **{len(missing)} of {plants_raw.shape[1]} columns** contain missing values.
> - Key columns used downstream with gaps: {_key_missing}.
> - Two numeric-looking columns hide **text sentinels** instead of true NaN:
>   `'Plant age'` contains `'unknown'`, and `'{CAPACITY_COL}'` contains `'>0'`
>   -- both are coerced with `pd.to_numeric(errors="coerce")` before use.
""".strip()))

13 of 44 columns contain missing values.



,n_missing,pct_missing
SOE status,1081,83.60
Other plant names (other language),958,74.09
Location address (other language),794,61.41
Owner (other language),715,55.30
Ferronickel capacity (ttpa),694,53.67
Coking plant capacity (ttpa),689,53.29
Other plant names (English),551,42.61
Pelletizing plant capacity (ttpa),528,40.84
Sinter plant capacity (ttpa),512,39.60
Plant name (other language),502,38.82


Completeness of the columns used downstream:
  GEM plant ID             missing:    0  (0.0%)
  Plant name (English)     missing:    0  (0.0%)
  Owner                    missing:    0  (0.0%)
  Country/area             missing:    0  (0.0%)
  Region                   missing:    0  (0.0%)
  Coordinates              missing:    0  (0.0%)
  Plant age                missing:   61  (4.7%)

Non-numeric sentinels found in numeric-looking columns:
  'Plant age' : ['unknown']
  'Nominal crude steel capacity (ttpa)' : ['>0']


> ### 🧹 Missing-data summary
> - **13 of 44 columns** contain missing values.
> - Key columns used downstream with gaps: **Plant age** (61).
> - Two numeric-looking columns hide **text sentinels** instead of true NaN:
>   `'Plant age'` contains `'unknown'`, and `'Nominal crude steel capacity (ttpa)'` contains `'>0'`
>   -- both are coerced with `pd.to_numeric(errors="coerce")` before use.

### Question 2: Statistical Summary
**Task:** Generate descriptive statistics for numerical columns.
- What is the average plant capacity? (sum relevant capacity columns in ttpa if needed, e.g. sinter / coking / pelletizing / ferronickel)
- What is the range of latitudes and longitudes? (you will likely need to parse `Coordinates` first — see Part 3 hint)
- What is the distribution of `Plant age (years)`?


In [41]:
# Display descriptive statistics
#
# Build the analysis frame first: parse coordinates, coerce the text sentinels
# ('unknown', '>0') to NaN, and aggregate unit-level capacity up to the plant.

def parse_coordinates(series: pd.Series) -> pd.DataFrame:
    """Split a 'lat, lon' text column into numeric Latitude / Longitude."""
    parts = series.astype("string").str.split(",", n=1, expand=True)
    # astype("float64") forces plain numpy floats. pandas 3 returns nullable
    # Float64 here, whose .to_numpy() yields an object array that numpy
    # ufuncs (np.radians in Part 4) cannot process.
    return pd.DataFrame({
        "Latitude": pd.to_numeric(parts[0].str.strip(), errors="coerce").astype("float64"),
        "Longitude": pd.to_numeric(parts[1].str.strip(), errors="coerce").astype("float64"),
    }, index=series.index)


def aggregate_capacity(units: pd.DataFrame) -> pd.DataFrame:
    """Sum operating crude-steel capacity per plant.

    A plant appears once per production unit. Only units that are currently
    operating count toward installed capacity, so announced / cancelled /
    retired units are excluded rather than silently summed in.
    """
    df = units.copy()
    df[CAPACITY_COL] = pd.to_numeric(df[CAPACITY_COL], errors="coerce").astype("float64")
    operating = df[df["Status"].isin(OPERATING_STATUSES)]
    return (
        operating.groupby("GEM plant ID")
        .agg(capacity_ttpa=(CAPACITY_COL, "sum"),
             n_operating_units=(CAPACITY_COL, "size"))
        .reset_index()
    )


plants = plants_raw.copy()
plants[["Latitude", "Longitude"]] = parse_coordinates(plants["Coordinates"])
plants["Plant age (years)"] = pd.to_numeric(plants["Plant age"], errors="coerce").astype("float64")

plants = plants.merge(aggregate_capacity(capacities_raw), on="GEM plant ID", how="left")
plants["n_operating_units"] = plants["n_operating_units"].fillna(0).astype(int)
plants["has_operating_capacity"] = plants["capacity_ttpa"].notna()

# Validate the parse rather than trusting it.
assert plants["Latitude"].between(-90, 90).all(), "Latitude out of range"
assert plants["Longitude"].between(-180, 180).all(), "Longitude out of range"
assert len(plants) == len(plants_raw), "Merge changed the row count"

print(f"Analysis frame: {plants.shape[0]:,} plants x {plants.shape[1]} columns")
print(f"  coordinates parsed     : {plants['Latitude'].notna().sum():,} / {len(plants):,}")
print(f"  with operating capacity: {plants['has_operating_capacity'].sum():,} "
      f"({plants['has_operating_capacity'].mean():.1%})")
print(f"  plant age known        : {plants['Plant age (years)'].notna().sum():,}\n")

summary_cols = ["capacity_ttpa", "Plant age (years)", "Latitude", "Longitude"]
display(plants[summary_cols].describe().T)

cap = plants["capacity_ttpa"]
age = plants["Plant age (years)"]
print(f"Average operating capacity : {cap.mean():,.0f} ttpa  (median {cap.median():,.0f} ttpa)")
print(f"Latitude range             : {plants['Latitude'].min():.2f} to {plants['Latitude'].max():.2f}")
print(f"Longitude range            : {plants['Longitude'].min():.2f} to {plants['Longitude'].max():.2f}")
print(f"Plant age                  : median {age.median():.0f} y, "
      f"mean {age.mean():.1f} y, max {age.max():.0f} y")

# The age distribution has a long right tail. These are not parsing errors:
# they are historic ironworks that are still operating sites, so "age" dates
# the SITE, not its current equipment.
print(f"\nPlants older than 100 years: {(age > 100).sum()} -- long-established sites, e.g.")
display(plants.nlargest(3, "Plant age (years)")[
    ["Plant name (English)", "Country/area", "Plant age (years)"]])

_oldest = plants.nlargest(1, "Plant age (years)").iloc[0]
display(Markdown(f"""
> ### 📈 Statistical summary
> | Metric | Value |
> |---|---|
> | **Average operating capacity** | **{cap.mean():,.0f} ttpa** (median {cap.median():,.0f} ttpa) |
> | Latitude range | {plants['Latitude'].min():.2f} to {plants['Latitude'].max():.2f} |
> | Longitude range | {plants['Longitude'].min():.2f} to {plants['Longitude'].max():.2f} |
> | **Plant age** | median **{age.median():.0f} y**, mean {age.mean():.1f} y, max {age.max():.0f} y |
> | Plants older than 100 years | **{(age > 100).sum()}** |
>
> The oldest tracked site is **{_oldest['Plant name (English)']}** ({_oldest['Country/area']},
> **{_oldest['Plant age (years)']:.0f} years**) -- a historic ironworks still in
> operation, so age here dates the *site*, not its current equipment.
""".strip()))

fig = px.histogram(
    plants[age.notna()], x="Plant age (years)", nbins=60, color="Region",
    title="Distribution of plant age by region",
)
fig.update_layout(height=430, bargap=0.05, yaxis_title="Number of plants")
fig.show()

Analysis frame: 1,293 plants x 50 columns
  coordinates parsed     : 1,293 / 1,293
  with operating capacity: 970 (75.0%)
  plant age known        : 1,125



,count,mean,std,min,25%,50%,75%,max
capacity_ttpa,970.00,"2,297.64","2,838.94",0.00,720.00,"1,200.00","2,795.00","22,999.00"
Plant age (years),"1,125.00",38.78,36.43,0.00,16.00,25.00,55.00,287.00
Latitude,"1,293.00",30.11,16.68,-37.83,23.50,33.96,39.98,67.19
Longitude,"1,293.00",64.23,66.40,-123.16,27.14,87.30,115.13,174.73


Average operating capacity : 2,298 ttpa  (median 1,200 ttpa)
Latitude range             : -37.83 to 67.19
Longitude range            : -123.16 to 174.73
Plant age                  : median 25 y, mean 38.8 y, max 287 y

Plants older than 100 years: 79 -- long-established sites, e.g.


,Plant name (English),Country/area,Plant age (years)
1125,TMK Seversky Pipe plant,Russia,287.00
820,OMK Vyksa Casting and Rolling Complex,Russia,268.99
371,GLGH Resita steel plant,Romania,255.00


> ### 📈 Statistical summary
> | Metric | Value |
> |---|---|
> | **Average operating capacity** | **2,298 ttpa** (median 1,200 ttpa) |
> | Latitude range | -37.83 to 67.19 |
> | Longitude range | -123.16 to 174.73 |
> | **Plant age** | median **25 y**, mean 38.8 y, max 287 y |
> | Plants older than 100 years | **79** |
>
> The oldest tracked site is **TMK Seversky Pipe plant** (Russia,
> **287 years**) -- a historic ironworks still in
> operation, so age here dates the *site*, not its current equipment.

### Question 3: Geographic Distribution
**Task:** Analyze the geographic distribution of steel plants.
- Which `Country/Area` or `Region` values have the most steel plants?
- What is the distribution of plants by `Owner` (company)?


In [42]:
# Count plants by country/region
by_country = (
    plants.groupby("Country/area")
    .agg(n_plants=("GEM plant ID", "size"),
         capacity_ttpa=("capacity_ttpa", "sum"))
    .sort_values("n_plants", ascending=False)
)
by_region = (
    plants.groupby("Region")
    .agg(n_plants=("GEM plant ID", "size"),
         capacity_ttpa=("capacity_ttpa", "sum"))
    .sort_values("n_plants", ascending=False)
)

print("Top 15 countries / areas by plant count:")
display(by_country.head(15))
print("Plants by region:")
display(by_region)

top_country = by_country.index[0]
print(f"{top_country} leads with {by_country.iloc[0]['n_plants']:,} plants "
      f"({by_country.iloc[0]['n_plants'] / len(plants):.1%} of the tracker).")

top_region = by_region.index[0]
_top3 = by_country.head(3)
_top3_md = " · ".join(f"**{c}** ({int(r['n_plants']):,})" for c, r in _top3.iterrows())
display(Markdown(f"""
> ### 🌍 Geographic distribution
> - Leading country/area: **{top_country}** with **{int(by_country.iloc[0]['n_plants']):,} plants**
>   ({by_country.iloc[0]['n_plants'] / len(plants):.1%} of the tracker).
> - Leading region: **{top_region}** with **{int(by_region.iloc[0]['n_plants']):,} plants**.
> - Top 3 countries/areas: {_top3_md}.
""".strip()))

fig = px.bar(
    by_country.head(15).reset_index(),
    x="n_plants", y="Country/area", orientation="h",
    color="n_plants", color_continuous_scale="Blues",
    labels={"n_plants": "Number of plants", "Country/area": ""},
    title="Top 15 countries / areas by number of steel plants",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"},
                  coloraxis_showscale=False, height=520)
fig.show()

Top 15 countries / areas by plant count:


,n_plants,capacity_ttpa
Country/area,,
China,458,"1,087,894.00"
India,113,"145,459.00"
United States,90,"112,196.00"
Iran,56,"36,758.00"
Japan,42,"105,900.00"
Russia,31,"85,015.00"
Türkiye,30,"56,332.00"
Vietnam,28,"39,926.00"
Brazil,25,"42,917.00"


Plants by region:


,n_plants,capacity_ttpa
Region,,
Asia Pacific,765,"1,557,791.00"
Europe,184,"266,764.00"
North America,113,"149,787.00"
Middle East,90,"68,038.00"
Africa,51,"38,390.00"
Eurasia,47,"92,915.00"
Central & South America,43,"55,028.00"


China leads with 458.0 plants (35.4% of the tracker).


> ### 🌍 Geographic distribution
> - Leading country/area: **China** with **458 plants**
>   (35.4% of the tracker).
> - Leading region: **Asia Pacific** with **765 plants**.
> - Top 3 countries/areas: **China** (458) · **India** (113) · **United States** (90).

In [43]:
# Count plants by Owner (company)
by_owner_count = (
    plants.groupby("Owner")
    .agg(n_plants=("GEM plant ID", "size"),
         n_countries=("Country/area", "nunique"))
    .sort_values("n_plants", ascending=False)
)

print(f"Distinct owners: {len(by_owner_count):,}")
print(f"Owners with a single plant: {(by_owner_count['n_plants'] == 1).sum():,} "
      f"({(by_owner_count['n_plants'] == 1).mean():.1%}) -> the ownership tail is long.\n")
print("Top 15 owners by number of plants:")
display(by_owner_count.head(15))

_top_owner = by_owner_count.index[0]
_top_owner_n_countries = int(by_owner_count.iloc[0]['n_countries'])
_country_word = "country/area" if _top_owner_n_countries == 1 else "countries/areas"
display(Markdown(f"""
> ### 🏢 Ownership distribution
> - **{len(by_owner_count):,} distinct owners** track the {len(plants):,} plants.
> - **{(by_owner_count['n_plants'] == 1).sum():,} owners ({(by_owner_count['n_plants'] == 1).mean():.1%})**
>   hold a single plant -- a long ownership tail.
> - Most active owner: **{_top_owner}** with **{int(by_owner_count.iloc[0]['n_plants']):,} plants**
>   across {_top_owner_n_countries} {_country_word}.
""".strip()))

fig = px.bar(
    by_owner_count.head(15).reset_index(),
    x="n_plants", y="Owner", orientation="h",
    color="n_plants", color_continuous_scale="Teal",
    labels={"n_plants": "Number of plants", "Owner": ""},
    title="Top 15 owners by number of steel plants",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"},
                  coloraxis_showscale=False, height=520)
fig.show()

Distinct owners: 1,069
Owners with a single plant: 962 (90.0%) -> the ownership tail is long.

Top 15 owners by number of plants:


,n_plants,n_countries
Owner,,
Nucor Corp,13,1
Cleveland-Cliffs Inc,12,2
Nippon Steel Corp,10,1
Commercial Metals Co,8,1
Steel Authority of India Ltd,8,1
Gerdau Ameristeel Corp,8,2
SteelAsia Manufacturing Corp,7,1
United States Steel Corp,6,1
ArcelorMittal Brasil SA,6,1


> ### 🏢 Ownership distribution
> - **1,069 distinct owners** track the 1,293 plants.
> - **962 owners (90.0%)**
>   hold a single plant -- a long ownership tail.
> - Most active owner: **Nucor Corp** with **13 plants**
>   across 1 country/area.

### Question 4: Capacity Analysis
**Task:** Analyze the capacity distribution.
- What is the total global steel production capacity? (sum the capacity columns you are using)
- Which `Owner` values have the highest total capacity?
- How does capacity vary by `Region` or `Country/Area`?


In [44]:
# Calculate total capacity
total_capacity_ttpa = plants["capacity_ttpa"].sum()

print("Global operating crude steel capacity")
print(f"  {total_capacity_ttpa:,.0f} ttpa")
print(f"  {total_capacity_ttpa / 1_000:,.1f} million tonnes per year")
print(f"  {total_capacity_ttpa / 1_000_000:,.2f} billion tonnes per year")
print(f"\nBased on {plants['has_operating_capacity'].sum():,} plants with at least one "
      f"operating unit (of {len(plants):,} tracked).")
print("Sanity check: world crude steel capacity is roughly 2.4 Gt/yr, so this is the "
      "right order of magnitude.")

print("\nCapacity by region:")
region_cap = (
    plants.groupby("Region")
    .agg(n_plants=("GEM plant ID", "size"), capacity_ttpa=("capacity_ttpa", "sum"))
    .sort_values("capacity_ttpa", ascending=False)
)
region_cap["share_pct"] = region_cap["capacity_ttpa"] / total_capacity_ttpa * 100
display(region_cap)

_top_region_cap = region_cap.iloc[0]
display(Markdown(f"""
> ### ⚙️ Global operating capacity
> ## **{total_capacity_ttpa / 1_000_000:,.2f} billion tonnes / year**
> ({total_capacity_ttpa:,.0f} ttpa, {total_capacity_ttpa / 1_000:,.1f} Mtpa)
>
> Based on **{plants['has_operating_capacity'].sum():,} plants** with at least one
> operating unit (of {len(plants):,} tracked). This matches the right order of
> magnitude for real-world global steel capacity (~2.4 Gt/yr).
>
> Leading region: **{region_cap.index[0]}** at **{_top_region_cap['share_pct']:.1f}%**
> of global capacity ({_top_region_cap['capacity_ttpa']:,.0f} ttpa).
""".strip()))

Global operating crude steel capacity
  2,228,713 ttpa
  2,228.7 million tonnes per year
  2.23 billion tonnes per year

Based on 970 plants with at least one operating unit (of 1,293 tracked).
Sanity check: world crude steel capacity is roughly 2.4 Gt/yr, so this is the right order of magnitude.

Capacity by region:


,n_plants,capacity_ttpa,share_pct
Region,,,
Asia Pacific,765,"1,557,791.00",69.90
Europe,184,"266,764.00",11.97
North America,113,"149,787.00",6.72
Eurasia,47,"92,915.00",4.17
Middle East,90,"68,038.00",3.05
Central & South America,43,"55,028.00",2.47
Africa,51,"38,390.00",1.72


> ### ⚙️ Global operating capacity
> ## **2.23 billion tonnes / year**
> (2,228,713 ttpa, 2,228.7 Mtpa)
>
> Based on **970 plants** with at least one
> operating unit (of 1,293 tracked). This matches the right order of
> magnitude for real-world global steel capacity (~2.4 Gt/yr).
>
> Leading region: **Asia Pacific** at **69.9%**
> of global capacity (1,557,791 ttpa).

In [45]:
# Group by Owner and sum capacity
by_owner_capacity = (
    plants.groupby("Owner")
    .agg(capacity_ttpa=("capacity_ttpa", "sum"),
         n_plants=("GEM plant ID", "size"),
         n_countries=("Country/area", "nunique"))
    .sort_values("capacity_ttpa", ascending=False)
)
by_owner_capacity["share_of_global_pct"] = (
    by_owner_capacity["capacity_ttpa"] / total_capacity_ttpa * 100
)

print("Top 15 owners by total operating capacity:")
display(by_owner_capacity.head(15))

top10_share = by_owner_capacity.head(10)["share_of_global_pct"].sum()
print(f"The 10 largest owners hold {top10_share:.1f}% of global operating capacity.")

_top_owner_cap = by_owner_capacity.iloc[0]
display(Markdown(f"""
> ### 🏆 Capacity concentration by owner
> - Largest owner by capacity: **{by_owner_capacity.index[0]}** with
>   **{_top_owner_cap['capacity_ttpa']:,.0f} ttpa** ({_top_owner_cap['share_of_global_pct']:.1f}% of global capacity).
> - The **top 10 owners** together hold **{top10_share:.1f}%** of global operating capacity.
""".strip()))

fig = px.bar(
    by_owner_capacity.head(15).reset_index(),
    x="capacity_ttpa", y="Owner", orientation="h",
    color="capacity_ttpa", color_continuous_scale="Oranges",
    labels={"capacity_ttpa": "Operating capacity (ttpa)", "Owner": ""},
    title="Top 15 owners by operating crude steel capacity",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"},
                  coloraxis_showscale=False, height=520)
fig.show()

# Capacity distribution across plants (log scale: the distribution is heavily skewed).
#
# NOTE: px.histogram bins nbins=60 LINEARLY over [min, max] before an axis is
# ever drawn. update_xaxes(type="log") only changes how that linear grid is
# rendered, not how it is computed -- with data spanning 35 to 22,999 ttpa,
# ~44% of plants fall into the first 3 (of 60) linear bins, and once those get
# squashed to the left edge of a log axis the chart reads as empty. Binning
# log10(capacity_ttpa) directly fixes this: bins are then evenly spaced in
# log space, matching what the log axis actually displays.
cap_pos = plants.loc[plants["capacity_ttpa"] > 0, "capacity_ttpa"]
log_cap = np.log10(cap_pos)

fig = px.histogram(
    x=log_cap, nbins=40,
    labels={"x": "Operating capacity (ttpa)"},
    title="Distribution of plant-level operating capacity",
)
tick_vals = np.arange(np.floor(log_cap.min()), np.ceil(log_cap.max()) + 1)
fig.update_xaxes(
    tickmode="array", tickvals=tick_vals,
    ticktext=[f"{10**v:,.0f}" for v in tick_vals],
    title="Operating capacity (ttpa, log scale)",
)
fig.update_layout(height=420, bargap=0.05, yaxis_title="Number of plants")
fig.show()

Top 15 owners by total operating capacity:


,capacity_ttpa,n_plants,n_countries,share_of_global_pct
Owner,,,,
Nippon Steel Corp,"44,423.00",10,1,1.99
POSCO Holdings Inc,"41,757.00",2,1,1.87
Angang Steel Co Ltd,"30,250.00",3,1,1.36
JFE Steel Corp,"28,768.00",5,1,1.29
JSW Steel Ltd,"28,359.00",5,1,1.27
Tata Steel Ltd,"26,460.00",6,2,1.19
Hyundai Steel Co,"24,297.00",4,2,1.09
Cleveland-Cliffs Inc,"23,655.00",12,2,1.06
Steel Authority of India Ltd,"21,350.00",8,1,0.96


The 10 largest owners hold 13.0% of global operating capacity.


> ### 🏆 Capacity concentration by owner
> - Largest owner by capacity: **Nippon Steel Corp** with
>   **44,423 ttpa** (2.0% of global capacity).
> - The **top 10 owners** together hold **13.0%** of global operating capacity.

---
## Part 3: Geospatial Visualization with Plotly

Create interactive maps to visualize the steel plants' locations and characteristics.


### Exercise 1: Basic Scatter Map
**Task:** Create a scatter map showing all steel plant locations.
- Parse `Coordinates` into numeric `Latitude` and `Longitude` (see hint in the code cell)
- Color points by `Country/Area` or `Region`
- Add hover information showing `Plant name (English)`, `Owner`, and capacity


In [46]:
# Create a scatter_geo or scatter_mapbox plot
#
# Coordinates were parsed in Part 2 (`parse_coordinates`): the source stores a
# single "lat, lon" string, now split into numeric Latitude / Longitude.

mappable = plants[plants["Latitude"].notna() & plants["Longitude"].notna()].copy()
mappable["capacity_label"] = mappable["capacity_ttpa"].map(
    lambda v: "no operating capacity" if pd.isna(v) else f"{v:,.0f} ttpa"
)
print(f"Plotting {len(mappable):,} plants with valid coordinates.")

fig = px.scatter_geo(
    mappable,
    lat="Latitude", lon="Longitude",
    color="Region",
    hover_name="Plant name (English)",
    hover_data={
        "Owner": True, "Country/area": True, "capacity_label": True,
        "Latitude": ":.3f", "Longitude": ":.3f", "Region": False,
    },
    projection="natural earth",
    title="Global steel plants by region",
)
fig.update_traces(marker=dict(size=5, opacity=0.75,
                              line=dict(width=0.3, color="white")))
fig.update_geos(showcountries=True, countrycolor="lightgray",
                showland=True, landcolor="#f7f7f7", showocean=True, oceancolor="#eaf3f8")
fig.update_layout(height=600, legend_title_text="Region")
fig.show()

Plotting 1,293 plants with valid coordinates.


### Exercise 2: Sized Markers by Capacity
**Task:** Create a map where marker size represents plant capacity.
- Larger markers for higher capacity plants
- Color by `Owner`
- Include interactive hover details (`Plant name (English)`, `Country/Area`, capacity, etc.)


In [47]:
# Create scatter map with size parameter based on capacity
#
# Only plants with operating capacity can be sized by it; plants without an
# operating unit are excluded here rather than drawn at an arbitrary size.

sized = mappable[mappable["capacity_ttpa"].notna() & (mappable["capacity_ttpa"] > 0)].copy()

# Restrict the colour legend to the largest owners so it stays readable.
TOP_N_OWNERS = 12
top_owners = (sized.groupby("Owner")["capacity_ttpa"].sum()
              .nlargest(TOP_N_OWNERS).index)
sized["Owner (grouped)"] = sized["Owner"].where(sized["Owner"].isin(top_owners), "Other owners")

print(f"Plotting {len(sized):,} plants with operating capacity > 0.")
print(f"Colouring the top {TOP_N_OWNERS} owners by capacity; the rest are 'Other owners'.")

fig = px.scatter_map(
    sized,
    lat="Latitude", lon="Longitude",
    size="capacity_ttpa", color="Owner (grouped)",
    size_max=38, zoom=1.1,
    hover_name="Plant name (English)",
    hover_data={
        "Owner": True, "Country/area": True, "capacity_ttpa": ":,.0f",
        "Plant age (years)": True, "Latitude": False, "Longitude": False,
        "Owner (grouped)": False,
    },
    map_style=MAP_STYLE,
    title="Steel plants sized by operating capacity, coloured by owner",
)
fig.update_layout(height=640, margin=dict(l=0, r=0, t=50, b=0),
                  legend_title_text="Owner")
fig.show()

Plotting 918 plants with operating capacity > 0.
Colouring the top 12 owners by capacity; the rest are 'Other owners'.


### Exercise 3: Density Heatmap
**Task:** Create a density map showing concentration of steel plants.
- Use Plotly's density_mapbox to show clustering
- Identify regions with high plant density


In [48]:
# Create density heatmap
fig = px.density_map(
    mappable,
    lat="Latitude", lon="Longitude",
    radius=18, zoom=1.1, center=dict(lat=28, lon=75),
    color_continuous_scale="Inferno",
    map_style=MAP_STYLE,
    title="Density of steel plants worldwide",
)
fig.update_layout(height=640, margin=dict(l=0, r=0, t=50, b=0),
                  coloraxis_colorbar_title="Plant<br>density")
fig.show()

# Quantify the clusters the heatmap shows.
cluster_tbl = (
    plants.groupby(["Region", "Country/area"])
    .agg(n_plants=("GEM plant ID", "size"), capacity_ttpa=("capacity_ttpa", "sum"))
    .sort_values("n_plants", ascending=False)
    .head(10)
)
print("Highest-density areas (top 10 country/area by plant count):")
display(cluster_tbl)

Highest-density areas (top 10 country/area by plant count):


n_plants  capacity_ttpa
Region                  Country/area                          
Asia Pacific            China               458   1,087,894.00
                        India               113     145,459.00
North America           United States        90     112,196.00
Middle East             Iran                 56      36,758.00
Asia Pacific            Japan                42     105,900.00
Eurasia                 Russia               31      85,015.00
Europe                  Türkiye              30      56,332.00
Asia Pacific            Vietnam              28      39,926.00
Central & South America Brazil               25      42,917.00
Europe                  Italy                24      31,736.00

---
## Part 4: Merging Exposure / Population Data with Assets

Steel plants sit in real places -- next to people, housing, and economic activity. In this part you will attach **granular socio-economic exposure data** to each plant so you can ask: *who and what is near this industrial asset?*

We use **LitPop** (ETH Zurich): a global dataset that combines **population** and **produced capital / asset value** on a fine geographic grid. It is widely used in disaster- and climate-risk analysis as a measure of **exposure**. It is **not** classical environmental monitoring (not air quality, emissions, or weather).

**Goal:** spatially link LitPop grid cells (or sample points) to steel plant locations (nearest neighbor or spatial join), then use the merged fields in maps and later company-level summaries.

> **Scope note (data available in this repo).** The LitPop samples in `litpop/` cover
> **China, India and Japan only** (300 arcsec, produced capital, USD 2018).
> Parts 1-3 above therefore use the **full global tracker (1,293 plants)**, while
> **Parts 4-5 and the exposure views of the dashboard are restricted to the 613 plants
> in those three countries** so that every exposure statistic is computed on complete
> data rather than on a half-empty column.

### Exercise 1: Load LitPop (Exposure) Data
**Task:** Load the LitPop sample and inspect it.

- **Samples for this lab (recommended):** LitPop sample files are available on Moodle (litpop data). Use these for the merge exercises below.
- **Full LitPop dataset (optional):** [ETH Research Collection – LitPop](https://www.research-collection.ethz.ch/entities/researchdata/12dcfc4f-9d03-463a-8d6b-76c0dc73cdc8)

- Expected columns (may vary by extract): location identifiers, latitude, longitude, population and/or asset-value / exposure fields, etc.

Briefly note what each column represents and the spatial resolution of the sample.


In [49]:
# Load LitPop sample (exposure / population-asset data)
#
# The lab's LitPop files are CLIMADA exposure objects stored as PyTables HDF5.
# They are read directly with h5py, which avoids pulling in the full CLIMADA stack.
#
# Layout inside each file, under the "exposures" group:
#   block0_values -> float columns  (value, latitude, longitude)
#   block1_values -> int columns    (region_id, impf_)
#   block2_values -> pickled geometry (not needed: lat/lon are already columns)

ISO3_TO_COUNTRY = {"CHN": "China", "IND": "India", "JPN": "Japan"}


def read_litpop_hdf5(path: Path) -> tuple[pd.DataFrame, dict]:
    """Read one CLIMADA LitPop HDF5 file into a tidy DataFrame + metadata."""
    with h5py.File(path, "r") as f:
        grp = f["exposures"]
        floats = grp["block0_values"][:]
        float_names = [c.decode() for c in grp["block0_items"][:]]
        ints = grp["block1_values"][:]
        int_names = [c.decode() for c in grp["block1_items"][:]]
        raw_meta = grp.attrs["metadata"]

    df = pd.DataFrame(floats, columns=float_names)
    for i, name in enumerate(int_names):
        df[name] = ints[:, i]

    # The metadata blob is a pickle referencing pyproj; read the fields we need
    # with a regex instead of unpickling (keeps the dependency list small).
    text = raw_meta.decode("latin1") if isinstance(raw_meta, bytes) else str(raw_meta)
    match = re.search(r"year: (\d{4}).*?financial mode: (\w+)", text)
    meta = {
        "ref_year": int(match.group(1)) if match else None,
        "fin_mode": match.group(2) if match else None,
        "value_unit": "USD",
        "resolution_arcsec": 300,
    }
    return df, meta


litpop_files = sorted(LITPOP_DIR.glob("LitPop_pc_300_arcsec_*_v1.hdf5"))
assert litpop_files, f"No LitPop files found in {LITPOP_DIR}"

frames, metas = [], {}
for path in litpop_files:
    iso3 = re.search(r"_([A-Z]{3})_v1", path.name).group(1)
    df, meta = read_litpop_hdf5(path)
    df["iso3"] = iso3
    df["country"] = ISO3_TO_COUNTRY.get(iso3, iso3)
    frames.append(df)
    metas[iso3] = meta
    print(f"{path.name:<42} {len(df):>7,} cells | total value {df['value'].sum():.3e} USD")

litpop = pd.concat(frames, ignore_index=True)
litpop = litpop.rename(columns={"value": "litpop_value_usd"})

print(f"\nCombined LitPop grid: {len(litpop):,} cells across {litpop['iso3'].nunique()} countries")
print(f"Metadata: {metas}")
litpop.head()

LitPop_pc_300_arcsec_CHN_v1.hdf5           136,991 cells | total value 3.932e+13 USD
LitPop_pc_300_arcsec_IND_v1.hdf5            40,101 cells | total value 6.840e+12 USD
LitPop_pc_300_arcsec_JPN_v1.hdf5             5,499 cells | total value 2.355e+13 USD

Combined LitPop grid: 182,591 cells across 3 countries
Metadata: {'CHN': {'ref_year': 2018, 'fin_mode': 'pc', 'value_unit': 'USD', 'resolution_arcsec': 300}, 'IND': {'ref_year': 2018, 'fin_mode': 'pc', 'value_unit': 'USD', 'resolution_arcsec': 300}, 'JPN': {'ref_year': 2018, 'fin_mode': 'pc', 'value_unit': 'USD', 'resolution_arcsec': 300}}


,litpop_value_usd,latitude,longitude,region_id,impf_,iso3,country
0,"5,280,439,584.86",20.04,110.21,156,1,CHN,China
1,"40,405,586.87",20.04,110.62,156,1,CHN,China
2,"41,902,243.33",20.04,110.71,156,1,CHN,China
3,"88,138,720.42",19.96,109.54,156,1,CHN,China
4,"187,994,709.31",19.96,109.62,156,1,CHN,China


In [50]:
# Inspect LitPop data (columns, dtypes, missing values, value ranges)
print("Columns and dtypes:")
display(litpop.dtypes.to_frame("dtype"))

print("Missing values per column:")
display(litpop.isna().sum().to_frame("n_missing"))

print("Value ranges by country:")
display(
    litpop.groupby("country").agg(
        n_cells=("litpop_value_usd", "size"),
        total_value_usd=("litpop_value_usd", "sum"),
        median_value_usd=("litpop_value_usd", "median"),
        max_value_usd=("litpop_value_usd", "max"),
        lat_min=("latitude", "min"), lat_max=("latitude", "max"),
        lon_min=("longitude", "min"), lon_max=("longitude", "max"),
    )
)

# Confirm the stated 300-arcsec resolution from the data itself.
step = np.median(np.diff(np.sort(litpop.loc[litpop.iso3 == "JPN", "latitude"].unique())))
print(f"Measured grid step: {step:.6f} deg = {step * 3600:.0f} arcsec "
      f"(~{step * 111:.1f} km at the equator)")

_by_country = litpop.groupby("country")["litpop_value_usd"].agg(["size", "sum"])
_cov_md = " · ".join(
    f"**{c}** ({n['size']:,} cells, ${n['sum']:.2e})" for c, n in _by_country.iterrows()
)
display(Markdown(f"""
> ### 🛰️ LitPop grid summary
> - **{len(litpop):,} grid cells** loaded across **{litpop['iso3'].nunique()} countries**: {_cov_md}.
> - Measured resolution: **{step * 3600:.0f} arcsec** (~{step * 111:.1f} km at the equator),
>   confirming the file name's stated "300 arcsec".
> - `litpop_value_usd` = **produced capital** (financial mode `pc`, ref. year 2018, USD) --
>   an asset-value proxy, **not** a population count.
> - **Coverage limit:** China, India and Japan only. Parts 4-5 below are scoped to these
>   three countries for that reason.
""".strip()))

Columns and dtypes:


,dtype
litpop_value_usd,float64
latitude,float64
longitude,float64
region_id,int64
impf_,int64
iso3,str
country,str


Missing values per column:


,n_missing
litpop_value_usd,0
latitude,0
longitude,0
region_id,0
impf_,0
iso3,0
country,0


Value ranges by country:


,n_cells,total_value_usd,median_value_usd,max_value_usd,lat_min,lat_max,lon_min,lon_max
country,,,,,,,,
China,136991,"39,322,741,137,081.70","379,089.39","287,347,665,233.51",18.21,53.54,73.62,134.71
India,40101,"6,840,055,978,929.30","14,459,652.18","51,926,786,868.75",6.88,35.46,68.21,97.29
Japan,5499,"23,547,820,098,991.80","24,605,676.26","504,405,695,271.74",24.29,45.46,122.96,145.79


Measured grid step: 0.083333 deg = 300 arcsec (~9.2 km at the equator)


> ### 🛰️ LitPop grid summary
> - **182,591 grid cells** loaded across **3 countries**: **China** (136,991.0 cells, $3.93e+13) · **India** (40,101.0 cells, $6.84e+12) · **Japan** (5,499.0 cells, $2.35e+13).
> - Measured resolution: **300 arcsec** (~9.2 km at the equator),
>   confirming the file name's stated "300 arcsec".
> - `litpop_value_usd` = **produced capital** (financial mode `pc`, ref. year 2018, USD) --
>   an asset-value proxy, **not** a population count.
> - **Coverage limit:** China, India and Japan only. Parts 4-5 below are scoped to these
>   three countries for that reason.

### Exercise 2: Spatial Join or Nearest Neighbor Matching
**Task:** Attach LitPop exposure attributes to each steel plant based on geographic proximity.
- Match each plant to the **nearest LitPop grid cell / sample point** (or use a spatial join if you work with polygons)
- Consider `geopandas`, a ball-tree / KD-tree nearest-neighbor search, or haversine distances
- Keep plant identifiers and the LitPop fields you will use later (e.g. population, asset value / exposure)

You should end up with one row per plant (or a clear many-to-one rule if you aggregate nearby cells).


In [51]:
# Calculate distances or perform spatial join
#
# Method: BallTree with the haversine metric (great-circle distance on a sphere).
# This is exact for lat/lon points and avoids the distortion of treating degrees
# as a plane -- important because the grid spans 18N to 54N.
#
# SCOPE: LitPop covers CHN / IND / JPN only, so the join is restricted to plants
# in those three countries. Parts 1-3 above used all 1,293 plants.

EARTH_RADIUS_KM = 6371.0088
NEIGHBOURHOOD_RADIUS_KM = 25.0   # "local context" window around each plant

covered_countries = sorted(ISO3_TO_COUNTRY.values())
plants_covered = plants[plants["Country/area"].isin(covered_countries)].copy()
plants_covered["iso3"] = plants_covered["Country/area"].map(
    {v: k for k, v in ISO3_TO_COUNTRY.items()}
)

print(f"LitPop coverage: {', '.join(covered_countries)}")
print(f"Plants in scope : {len(plants_covered):,} of {len(plants):,} "
      f"({len(plants_covered) / len(plants):.1%} of the tracker)")
print(plants_covered["Country/area"].value_counts().to_string(), end="\n\n")


def match_plants_to_litpop(plants_df: pd.DataFrame, grid_df: pd.DataFrame) -> pd.DataFrame:
    """Nearest LitPop cell per plant + summed exposure within a radius.

    One tree per country: a plant is only ever matched to a cell in its own
    country, which prevents cross-border matches along coastlines.
    """
    results = []
    for iso3, group in plants_df.groupby("iso3"):
        grid = grid_df[grid_df["iso3"] == iso3].reset_index(drop=True)
        # dtype="float64" guards against pandas nullable dtypes reaching numpy.
        tree = BallTree(
            np.radians(grid[["latitude", "longitude"]].to_numpy(dtype="float64")),
            metric="haversine",
        )
        pts = np.radians(group[["Latitude", "Longitude"]].to_numpy(dtype="float64"))

        dist, idx = tree.query(pts, k=1)                       # nearest cell
        within = tree.query_radius(pts, r=NEIGHBOURHOOD_RADIUS_KM / EARTH_RADIUS_KM)

        values = grid["litpop_value_usd"].to_numpy(dtype="float64")
        out = group.copy()
        out["litpop_value_usd"] = values[idx[:, 0]]
        out["litpop_cell_lat"] = grid["latitude"].to_numpy(dtype="float64")[idx[:, 0]]
        out["litpop_cell_lon"] = grid["longitude"].to_numpy(dtype="float64")[idx[:, 0]]
        out["match_distance_km"] = dist[:, 0] * EARTH_RADIUS_KM
        out[f"litpop_sum_{int(NEIGHBOURHOOD_RADIUS_KM)}km_usd"] = [values[i].sum() for i in within]
        out[f"litpop_ncells_{int(NEIGHBOURHOOD_RADIUS_KM)}km"] = [len(i) for i in within]
        results.append(out)
    return pd.concat(results).sort_index()


plants_litpop = match_plants_to_litpop(plants_covered, litpop)

print("Match quality (distance from plant to the centroid of its nearest cell):")
display(plants_litpop["match_distance_km"].describe().to_frame("match_distance_km"))
print(f"Cell size is ~10 km, so a median of "
      f"{plants_litpop['match_distance_km'].median():.1f} km is the expected magnitude.")
print(f"Matches further than 15 km: {(plants_litpop['match_distance_km'] > 15).sum()} "
      f"(inspect these before trusting them)")
display(plants_litpop.nlargest(3, "match_distance_km")[
    ["Plant name (English)", "Country/area", "match_distance_km", "litpop_value_usd"]])

_dist_stats = plants_litpop["match_distance_km"].describe()
display(Markdown(f"""
> ### 🎯 Spatial join quality
> - **{len(plants_litpop):,} plants** matched, one nearest LitPop cell each.
> - Median match distance: **{_dist_stats['50%']:.1f} km** (mean {_dist_stats['mean']:.1f} km,
>   max {_dist_stats['max']:.1f} km) -- consistent with a ~10 km grid cell.
> - **{(plants_litpop['match_distance_km'] > 15).sum()} match(es)** exceed 15 km and were
>   inspected individually above.
""".strip()))

LitPop coverage: China, India, Japan
Plants in scope : 613 of 1,293 (47.4% of the tracker)
Country/area
China    458
India    113
Japan     42

Match quality (distance from plant to the centroid of its nearest cell):


,match_distance_km
count,613.00
mean,3.44
std,1.62
min,0.03
25%,2.33
50%,3.53
75%,4.38
max,18.27


Cell size is ~10 km, so a median of 3.5 km is the expected magnitude.
Matches further than 15 km: 1 (inspect these before trusting them)


,Plant name (English),Country/area,match_distance_km,litpop_value_usd
1287,Zhuhai Yueyufeng Iron and Steel Co Ltd,China,18.27,"2,524,762,539.68"
1278,Zenith Steel Group Nantong Co Ltd,China,10.61,"1,230,988,127.12"
332,Fujian Qingtuo New Material Co Ltd,China,9.86,"11,794,218.39"


> ### 🎯 Spatial join quality
> - **613 plants** matched, one nearest LitPop cell each.
> - Median match distance: **3.5 km** (mean 3.4 km,
>   max 18.3 km) -- consistent with a ~10 km grid cell.
> - **1 match(es)** exceed 15 km and were
>   inspected individually above.

In [52]:
# Merge datasets
EXPOSURE_NEAREST = "litpop_value_usd"
EXPOSURE_LOCAL = f"litpop_sum_{int(NEIGHBOURHOOD_RADIUS_KM)}km_usd"

keep_cols = [
    "GEM plant ID", "Plant name (English)", "Owner", "Parent (English)",
    "Country/area", "Region", "iso3", "Latitude", "Longitude",
    "Plant age (years)", "capacity_ttpa", "n_operating_units",
    "Main production equipment", "Status" if "Status" in plants_litpop.columns else "iso3",
    EXPOSURE_NEAREST, EXPOSURE_LOCAL,
    f"litpop_ncells_{int(NEIGHBOURHOOD_RADIUS_KM)}km",
    "litpop_cell_lat", "litpop_cell_lon", "match_distance_km",
]
keep_cols = list(dict.fromkeys(c for c in keep_cols if c in plants_litpop.columns))

plants_merged = plants_litpop[keep_cols].copy()

# One row per plant, and every plant matched.
assert plants_merged["GEM plant ID"].is_unique, "Merge produced duplicate plants"
assert plants_merged[EXPOSURE_NEAREST].notna().all(), "Unmatched plants remain"

print(f"Merged dataset: {plants_merged.shape[0]:,} plants x {plants_merged.shape[1]} columns")
print(f"  every plant matched to a LitPop cell : {plants_merged[EXPOSURE_NEAREST].notna().all()}")
print(f"  plants with operating capacity       : {plants_merged['capacity_ttpa'].notna().sum():,}")

print("\nExposure summary by country:")
_exposure_by_country = plants_merged.groupby("Country/area").agg(
    n_plants=("GEM plant ID", "size"),
    capacity_ttpa=("capacity_ttpa", "sum"),
    median_nearest_exposure_usd=(EXPOSURE_NEAREST, "median"),
    median_local_exposure_usd=(EXPOSURE_LOCAL, "median"),
    median_match_km=("match_distance_km", "median"),
)
display(_exposure_by_country)

display(Markdown(f"""
> ### 🔗 Merged plant + exposure dataset
> - **{plants_merged.shape[0]:,} plants x {plants_merged.shape[1]} columns**, one row per plant.
> - **100% matched**: every plant has a LitPop exposure value (verified by assertion).
> - **{plants_merged['capacity_ttpa'].notna().sum():,}** of these plants also report operating capacity.
> - Median local exposure (25 km) ranges from
>   **${_exposure_by_country['median_local_exposure_usd'].min():.2e}** to
>   **${_exposure_by_country['median_local_exposure_usd'].max():.2e}** across the three countries.
""".strip()))
plants_merged.head()

Merged dataset: 613 plants x 19 columns
  every plant matched to a LitPop cell : True
  plants with operating capacity       : 481

Exposure summary by country:


,n_plants,capacity_ttpa,median_nearest_exposure_usd,median_local_exposure_usd,median_match_km
Country/area,,,,,
China,458,"1,087,894.00","793,380,922.64","18,676,345,030.98",3.52
India,113,"145,459.00","450,845,929.99","5,832,849,600.81",3.55
Japan,42,"105,900.00","12,824,930,411.58","172,814,499,967.08",3.75


> ### 🔗 Merged plant + exposure dataset
> - **613 plants x 19 columns**, one row per plant.
> - **100% matched**: every plant has a LitPop exposure value (verified by assertion).
> - **481** of these plants also report operating capacity.
> - Median local exposure (25 km) ranges from
>   **$5.83e+09** to
>   **$1.73e+11** across the three countries.

,GEM plant ID,Plant name (English),Owner,Parent (English),Country/area,Region,iso3,Latitude,Longitude,Plant age (years),capacity_ttpa,n_operating_units,Main production equipment,litpop_value_usd,litpop_sum_25km_usd,litpop_ncells_25km,litpop_cell_lat,litpop_cell_lon,match_distance_km
12,P100001045296,Action Ispat and Power Jharsuguda steel plant,Action Ispat and Power Pvt Ltd,Action Ispat and Power Pvt Ltd [100.0%],India,Asia Pacific,IND,21.84,83.98,NaN,375.00,1,DRI; IF; Steel other/unspecified,"645,075,019.35","6,421,397,393.39",26,21.88,83.96,4.63
13,P100001025815,Adhunik Metaliks Kuanrmunda steel plant,Adhunik Metaliks Ltd,natural person(s) [36.1%]; small shareholder(s...,India,Asia Pacific,IND,22.31,84.76,22.81,500.00,1,BF; DRI; EAF,"721,177,173.75","9,523,780,056.89",25,22.29,84.79,3.43
21,P100000120644,Aichi Steel Chita plant (Tokai),Aichi Steel Corp,Aichi Steel Corp [100.0%],Japan,Asia Pacific,JPN,35.04,136.90,83.00,"1,495.00",1,EAF,"110,186,877,555.41","1,502,462,582,382.96",21,35.04,136.96,5.25
42,P100000120163,Angang Group Xinyang Iron and Steel Co Ltd,"Xinyang Iron & Steel Co.,ltd of Angang Group",natural person(s) [65.2%]; Henan PROVINCE HENG...,China,Asia Pacific,CHN,32.49,114.04,55.00,"3,600.00",1,BF; BOF,"382,250,327.83","823,121,774.58",28,32.46,114.04,3.39
43,P100000120089,Angang Lianzhong Stainless Steel Co Ltd,Angang Lianzhong Stainless Steel Corp,Ansteel Group Co Ltd [60.0%]; Lianzhong Co Ltd...,China,Asia Pacific,CHN,23.13,113.51,25.00,"1,900.00",1,EAF,"23,605,391,215.71","860,628,677,784.88",25,23.12,113.54,3.67


### Exercise 3: Visualize Plants with Exposure Context
**Task:** Create a map of steel plants enriched with LitPop fields.
- Color plants by a LitPop metric (e.g. local population or asset exposure)
- Size markers by plant capacity
- Add hover details with both plant attributes (`Plant name (English)`, `Owner`, capacity) and the matched LitPop values

Interpret briefly: where do large plants sit relative to high population / high asset-value areas?


In [53]:
# Create visualization of plants colored by LitPop exposure metrics
#
# Exposure is extremely skewed (city centres dwarf rural cells), so colour is
# mapped on a log10 scale -- otherwise every point but a handful looks identical.

viz = plants_merged[plants_merged["capacity_ttpa"].notna()
                    & (plants_merged["capacity_ttpa"] > 0)].copy()
viz["log10_local_exposure"] = np.log10(viz[EXPOSURE_LOCAL])
viz["local_exposure_busd"] = viz[EXPOSURE_LOCAL] / 1e9

print(f"Mapping {len(viz):,} plants with operating capacity in {', '.join(covered_countries)}.")

fig = px.scatter_map(
    viz,
    lat="Latitude", lon="Longitude",
    color="log10_local_exposure", size="capacity_ttpa",
    size_max=34, zoom=2.4, center=dict(lat=30, lon=100),
    color_continuous_scale="Turbo",
    hover_name="Plant name (English)",
    hover_data={
        "Owner": True, "Country/area": True,
        "capacity_ttpa": ":,.0f", "local_exposure_busd": ":,.1f",
        "match_distance_km": ":.1f",
        "log10_local_exposure": False, "Latitude": False, "Longitude": False,
    },
    labels={"capacity_ttpa": "Capacity (ttpa)",
            "local_exposure_busd": f"Exposure within {int(NEIGHBOURHOOD_RADIUS_KM)} km (bn USD)"},
    map_style=MAP_STYLE,
    title=(f"Steel plants sized by capacity, coloured by LitPop asset exposure "
           f"within {int(NEIGHBOURHOOD_RADIUS_KM)} km (log10 USD, 2018)"),
)
fig.update_layout(height=660, margin=dict(l=0, r=0, t=50, b=0),
                  coloraxis_colorbar_title="log10<br>USD")
fig.show()

# Is plant size related to how built-up its surroundings are?
corr = np.corrcoef(np.log10(viz["capacity_ttpa"]), viz["log10_local_exposure"])[0, 1]
fig = px.scatter(
    viz, x=EXPOSURE_LOCAL, y="capacity_ttpa", color="Country/area",
    hover_name="Plant name (English)", log_x=True, log_y=True,
    labels={EXPOSURE_LOCAL: f"LitPop exposure within {int(NEIGHBOURHOOD_RADIUS_KM)} km (USD)",
            "capacity_ttpa": "Operating capacity (ttpa)"},
    title=f"Plant capacity vs local asset exposure (log-log, Pearson r = {corr:.2f})",
)
fig.update_layout(height=480)
fig.show()

q = viz[EXPOSURE_LOCAL].quantile([0.25, 0.75])
high, low = viz[viz[EXPOSURE_LOCAL] >= q[0.75]], viz[viz[EXPOSURE_LOCAL] <= q[0.25]]
print(f"""
Interpretation
--------------
Correlation between log capacity and log local exposure: r = {corr:.2f}.
Plants in the TOP exposure quartile  : {len(high):>3}, mean capacity {high['capacity_ttpa'].mean():>8,.0f} ttpa
Plants in the BOTTOM exposure quartile: {len(low):>3}, mean capacity {low['capacity_ttpa'].mean():>8,.0f} ttpa

Large plants sit in visibly built-up corridors -- the Chinese coastal belt
(Hebei/Jiangsu/Liaoning), the Indian mineral belt around Jharkhand/Odisha, and
the Japanese Pacific industrial coast. High-exposure surroundings mean a
disruption at these sites (or to these sites) coincides with dense
population and capital, which is exactly the framing disaster-risk analysis uses.
""")

Mapping 451 plants with operating capacity in China, India, Japan.



Interpretation
--------------
Correlation between log capacity and log local exposure: r = 0.12.
Plants in the TOP exposure quartile  : 113, mean capacity    3,624 ttpa
Plants in the BOTTOM exposure quartile: 113, mean capacity    2,324 ttpa

Large plants sit in visibly built-up corridors -- the Chinese coastal belt
(Hebei/Jiangsu/Liaoning), the Indian mineral belt around Jharkhand/Odisha, and
the Japanese Pacific industrial coast. High-exposure surroundings mean a
disruption at these sites (or to these sites) coincides with dense
population and capital, which is exactly the framing disaster-risk analysis uses.



---
## Part 5: Company-Level Aggregation

Aggregate data at the company level to analyze corporate footprints — including capacity and the LitPop exposure context you attached in Part 4.


### Exercise 1: Aggregate Metrics by Company
**Task:** Group plants by company (`Owner`) and calculate aggregate metrics.
- Total capacity per company
- Number of plants per company
- Average LitPop exposure metrics per company (from Part 4)
- Geographic spread (e.g. number of `Country/Area` or `Region` values)


In [54]:
# Group by company and aggregate
#
# Scope note: company aggregates are computed on the LitPop-covered subset
# (CHN / IND / JPN), so every exposure average is backed by real data rather
# than a partially-empty column.

companies = (
    plants_merged.groupby("Owner")
    .agg(
        n_plants=("GEM plant ID", "size"),
        total_capacity_ttpa=("capacity_ttpa", "sum"),
        mean_capacity_ttpa=("capacity_ttpa", "mean"),
        max_capacity_ttpa=("capacity_ttpa", "max"),
        n_countries=("Country/area", "nunique"),
        n_regions=("Region", "nunique"),
        mean_plant_age=("Plant age (years)", "mean"),
        mean_nearest_exposure_usd=(EXPOSURE_NEAREST, "mean"),
        mean_local_exposure_usd=(EXPOSURE_LOCAL, "mean"),
        max_local_exposure_usd=(EXPOSURE_LOCAL, "max"),
    )
    .reset_index()
)

# Capacity-weighted exposure: a company's risk profile should be driven by where
# its BIG plants are, not by an unweighted average over sites of any size.
weighted = (
    plants_merged.dropna(subset=["capacity_ttpa"])
    .groupby("Owner")
    .apply(lambda g: np.average(g[EXPOSURE_LOCAL], weights=g["capacity_ttpa"])
           if g["capacity_ttpa"].sum() > 0 else np.nan,
           include_groups=False)
    .rename("capacity_weighted_exposure_usd")
    .reset_index()
)
companies = companies.merge(weighted, on="Owner", how="left")
companies["countries"] = (
    plants_merged.groupby("Owner")["Country/area"]
    .agg(lambda s: ", ".join(sorted(s.unique()))).values
)
companies = companies.sort_values("total_capacity_ttpa", ascending=False).reset_index(drop=True)

print(f"Companies (owners) in the covered subset: {len(companies):,}")
print(f"  multi-plant companies : {(companies['n_plants'] > 1).sum():,}")
print(f"  multi-country companies: {(companies['n_countries'] > 1).sum():,}\n")
print("Top 15 companies by total operating capacity:")
display(companies.head(15)[[
    "Owner", "n_plants", "total_capacity_ttpa", "n_countries",
    "mean_plant_age", "mean_local_exposure_usd", "capacity_weighted_exposure_usd",
]])

_top_company = companies.iloc[0]
display(Markdown(f"""
> ### 🏭 Company-level aggregation
> - **{len(companies):,} companies** (owners) in the LitPop-covered subset.
> - **{(companies['n_plants'] > 1).sum():,}** are multi-plant, **{(companies['n_countries'] > 1).sum():,}** multi-country.
> - Largest company by total capacity: **{_top_company['Owner']}** with
>   **{_top_company['total_capacity_ttpa']:,.0f} ttpa** across **{_top_company['n_plants']:,} plants**.
""".strip()))

Companies (owners) in the covered subset: 535
  multi-plant companies : 38
  multi-country companies: 0

Top 15 companies by total operating capacity:


,Owner,n_plants,total_capacity_ttpa,n_countries,mean_plant_age,mean_local_exposure_usd,capacity_weighted_exposure_usd
0,Nippon Steel Corp,10,"44,423.00",1,79.35,"257,528,786,379.09","319,517,129,600.17"
1,Angang Steel Co Ltd,3,"30,250.00",1,22.01,"74,041,047,571.84","128,072,046,899.56"
2,JFE Steel Corp,5,"28,768.00",1,62.49,"890,055,524,489.19","206,868,809,109.77"
3,JSW Steel Ltd,5,"28,359.00",1,28.25,"4,461,305,434.65","4,412,760,509.52"
4,Tata Steel Ltd,5,"25,710.00",1,49.50,"15,876,852,892.78","13,955,362,893.06"
5,Steel Authority of India Ltd,8,"21,350.00",1,70.69,"23,408,073,014.27","24,605,050,712.24"
6,Baoshan Iron & Steel Co Ltd,1,"19,800.00",1,49.00,"1,390,784,081,982.44","1,390,784,081,982.44"
7,Maanshan Iron & Steel Co Ltd,1,"17,340.00",1,73.00,"67,473,115,662.36","67,473,115,662.36"
8,Jiangsu Shagang Iron & Steel Co Ltd,1,"17,305.00",1,23.60,"157,366,677,006.87","157,366,677,006.87"
9,Inner Mongolia BaoTou Steel Union Co Ltd,1,"16,760.00",1,26.64,"145,202,593,779.92","145,202,593,779.92"


> ### 🏭 Company-level aggregation
> - **535 companies** (owners) in the LitPop-covered subset.
> - **38** are multi-plant, **0** multi-country.
> - Largest company by total capacity: **Nippon Steel Corp** with
>   **44,423 ttpa** across **10 plants**.

### Exercise 2: Company Headquarters or Centroid
**Task:** Calculate a representative location for each company.
- Option 1: Use the centroid of all plant locations
- Option 2: Use the location of the largest plant
- Option 3: Assign actual headquarters coordinates


In [55]:
# Calculate company representative locations
#
# Option 1 (centroid) and Option 2 (largest plant) are both computed. The
# centroid is a spherical mean -- averaging degrees directly is wrong near the
# antimeridian, and is inaccurate even away from it.

def spherical_centroid(lat_deg: np.ndarray, lon_deg: np.ndarray) -> tuple[float, float]:
    """Mean position on a sphere, via 3-D unit vectors."""
    lat, lon = np.radians(lat_deg), np.radians(lon_deg)
    x = np.mean(np.cos(lat) * np.cos(lon))
    y = np.mean(np.cos(lat) * np.sin(lon))
    z = np.mean(np.sin(lat))
    return (np.degrees(np.arctan2(z, np.hypot(x, y))),
            np.degrees(np.arctan2(y, x)))


centroids = (
    plants_merged.groupby("Owner")
    .apply(lambda g: pd.Series(spherical_centroid(g["Latitude"].to_numpy(dtype="float64"),
                                                  g["Longitude"].to_numpy(dtype="float64")),
                               index=["centroid_lat", "centroid_lon"]),
           include_groups=False)
    .reset_index()
)

# Option 2: the company's largest plant by capacity (NaN capacity sorts last).
largest = (
    plants_merged.sort_values("capacity_ttpa", ascending=False, na_position="last")
    .groupby("Owner", as_index=False)
    .first()[["Owner", "Plant name (English)", "Latitude", "Longitude", "capacity_ttpa"]]
    .rename(columns={"Plant name (English)": "largest_plant",
                     "Latitude": "largest_plant_lat",
                     "Longitude": "largest_plant_lon",
                     "capacity_ttpa": "largest_plant_ttpa"})
)

companies = companies.merge(centroids, on="Owner", how="left").merge(largest, on="Owner", how="left")

# Representative point used for mapping: the largest plant where one exists
# (a real place), otherwise the centroid.
companies["rep_lat"] = companies["largest_plant_lat"].fillna(companies["centroid_lat"])
companies["rep_lon"] = companies["largest_plant_lon"].fillna(companies["centroid_lon"])
companies["rep_source"] = np.where(companies["largest_plant_lat"].notna(),
                                   "largest plant", "centroid")

assert companies["rep_lat"].between(-90, 90).all()
assert companies["rep_lon"].between(-180, 180).all()

print("Representative location source:")
print(companies["rep_source"].value_counts().to_string(), end="\n\n")

# For single-plant companies the two options coincide -- check the multi-plant ones.
multi = companies[companies["n_plants"] > 1]
spread_km = EARTH_RADIUS_KM * np.radians(
    np.hypot(multi["centroid_lat"] - multi["largest_plant_lat"],
             (multi["centroid_lon"] - multi["largest_plant_lon"])
             * np.cos(np.radians(multi["centroid_lat"]))))
print(f"For the {len(multi):,} multi-plant companies, centroid and largest-plant "
      f"locations differ by a median of {spread_km.median():,.0f} km "
      f"(max {spread_km.max():,.0f} km).")
display(companies.head(10)[[
    "Owner", "n_plants", "centroid_lat", "centroid_lon",
    "largest_plant", "largest_plant_lat", "largest_plant_lon", "rep_source"]])

_rep_counts = companies["rep_source"].value_counts()
display(Markdown(f"""
> ### 📍 Company representative locations
> - **{_rep_counts.get('largest plant', 0):,}** companies represented by their **largest plant**,
>   **{_rep_counts.get('centroid', 0):,}** by a **spherical centroid** (single-plant companies, where the two coincide).
> - For the **{len(multi):,} multi-plant companies**, centroid vs. largest-plant location
>   differs by a median of **{spread_km.median():,.0f} km** (max {spread_km.max():,.0f} km) --
>   a reminder that "one point per company" hides real geographic spread.
""".strip()))

Representative location source:
rep_source
largest plant    535

For the 38 multi-plant companies, centroid and largest-plant locations differ by a median of 72 km (max 951 km).


,Owner,n_plants,centroid_lat,centroid_lon,largest_plant,largest_plant_lat,largest_plant_lon,rep_source
0,Nippon Steel Corp,10,35.36,135.32,Nippon East Japan Works (Kimitsu) steel plant,35.36,139.88,largest plant
1,Angang Steel Co Ltd,3,41.00,121.83,Angang Steel Co Ltd,41.15,122.98,largest plant
2,JFE Steel Corp,5,35.70,137.56,JFE West Japan Works (Fukuyama) steel plant,34.47,133.43,largest plant
3,JSW Steel Ltd,5,17.56,78.29,JSW Steel Vijayanagar steel plant,15.18,76.66,largest plant
4,Tata Steel Ltd,5,23.71,84.03,Tata Steel Jamshedpur steel plant,22.79,86.20,largest plant
5,Steel Authority of India Ltd,8,22.69,85.71,SAIL Bhilai steel plant,21.19,81.39,largest plant
6,Baoshan Iron & Steel Co Ltd,1,31.42,121.44,Baoshan Iron and Steel Co Ltd Headquarters,31.42,121.44,largest plant
7,Maanshan Iron & Steel Co Ltd,1,31.70,118.47,Maanshan Iron and Steel Co Ltd,31.70,118.47,largest plant
8,Jiangsu Shagang Iron & Steel Co Ltd,1,31.98,120.64,Jiangsu Shagang Iron & Steel Co Ltd,31.98,120.64,largest plant
9,Inner Mongolia BaoTou Steel Union Co Ltd,1,40.65,109.74,Inner Mongolia BaoTou Steel Union Co Ltd,40.65,109.74,largest plant


> ### 📍 Company representative locations
> - **535** companies represented by their **largest plant**,
>   **0** by a **spherical centroid** (single-plant companies, where the two coincide).
> - For the **38 multi-plant companies**, centroid vs. largest-plant location
>   differs by a median of **72 km** (max 951 km) --
>   a reminder that "one point per company" hides real geographic spread.

### Exercise 3: Visualize Company-Level Data
**Task:** Create a map showing companies with aggregated metrics.
- Show one marker per company at the representative location
- Size by total capacity
- Color by average LitPop exposure (or another Part 4 metric)
- Hover information with company summary statistics


In [56]:
# Create company-level visualization
TOP_N_COMPANIES = 60

plot_df = (companies[companies["total_capacity_ttpa"] > 0]
           .nlargest(TOP_N_COMPANIES, "total_capacity_ttpa").copy())
plot_df["log10_exposure"] = np.log10(plot_df["mean_local_exposure_usd"])
plot_df["capacity_mtpa"] = plot_df["total_capacity_ttpa"] / 1_000
plot_df["mean_exposure_busd"] = plot_df["mean_local_exposure_usd"] / 1e9

fig = px.scatter_map(
    plot_df,
    lat="rep_lat", lon="rep_lon",
    size="total_capacity_ttpa", color="log10_exposure",
    size_max=45, zoom=2.4, center=dict(lat=30, lon=100),
    color_continuous_scale="Turbo",
    hover_name="Owner",
    hover_data={
        "n_plants": True, "capacity_mtpa": ":,.1f", "n_countries": True,
        "countries": True, "mean_exposure_busd": ":,.1f",
        "mean_plant_age": ":.0f", "rep_source": True,
        "log10_exposure": False, "rep_lat": False, "rep_lon": False,
        "total_capacity_ttpa": False,
    },
    labels={"n_plants": "Plants", "capacity_mtpa": "Total capacity (Mtpa)",
            "n_countries": "Countries", "mean_exposure_busd": "Mean local exposure (bn USD)",
            "mean_plant_age": "Mean plant age (y)", "rep_source": "Marker located at"},
    map_style=MAP_STYLE,
    title=(f"Top {TOP_N_COMPANIES} steel companies: one marker per company, "
           f"sized by total capacity, coloured by mean local LitPop exposure"),
)
fig.update_layout(height=680, margin=dict(l=0, r=0, t=50, b=0),
                  coloraxis_colorbar_title="log10<br>USD")
fig.show()

# Capacity vs exposure positioning for the largest companies.
fig = px.scatter(
    plot_df, x="mean_local_exposure_usd", y="total_capacity_ttpa",
    size="n_plants", color="n_countries", hover_name="Owner",
    log_x=True, log_y=True, size_max=32, color_continuous_scale="Viridis",
    labels={"mean_local_exposure_usd": f"Mean LitPop exposure within {int(NEIGHBOURHOOD_RADIUS_KM)} km (USD)",
            "total_capacity_ttpa": "Total operating capacity (ttpa)",
            "n_countries": "Countries", "n_plants": "Plants"},
    title=f"Top {TOP_N_COMPANIES} companies: capacity vs surrounding asset exposure",
)
fig.update_layout(height=520)
fig.show()

print("Companies whose portfolio sits in the highest-exposure surroundings "
      "(capacity-weighted, min. 2 plants):")
display(
    companies[companies["n_plants"] >= 2]
    .nlargest(10, "capacity_weighted_exposure_usd")[[
        "Owner", "n_plants", "total_capacity_ttpa",
        "capacity_weighted_exposure_usd", "countries"]]
)

Companies whose portfolio sits in the highest-exposure surroundings (capacity-weighted, min. 2 plants):


,Owner,n_plants,total_capacity_ttpa,capacity_weighted_exposure_usd,countries
221,Godo Steel Ltd,3,"1,700.00","2,534,076,359,059.48",Japan
160,Kyoei Steel Ltd,3,"2,645.00","1,472,531,591,936.45",Japan
0,Nippon Steel Corp,10,"44,423.00","319,517,129,600.17",Japan
46,Kobe Steel Ltd,2,"6,591.00","254,696,791,275.57",Japan
2,JFE Steel Corp,5,"28,768.00","206,868,809,109.77",Japan
76,Tokyo Steel Manufacturing Co Ltd,4,"4,786.00","149,183,166,332.32",Japan
1,Angang Steel Co Ltd,3,"30,250.00","128,072,046,899.56",China
184,JFE Bars & Shapes Corp,3,"2,220.00","95,015,461,029.15",Japan
118,Xingtai Iron and Steel Co Ltd,2,"3,060.00","74,501,064,540.98",China
402,Captain Steel India Ltd,2,208.00,"57,609,016,229.04",India


---
## Part 6: Streamlit Dashboard Integration

Prepare your visualizations for deployment in a Streamlit dashboard.


### Exercise 1: Create Dashboard Script Structure
**Task:** Create a Streamlit app file (`app.py`) with the following structure:

```python
# Import streamlit and other necessary libraries

# Set page configuration

# Title and description

# Sidebar for filters
# - Company selector
# - Region/country filter
# - Capacity range slider

# Main content area
# - KPI metrics (total plants, total capacity, etc.)
# - Interactive map
# - Data table

# Footer with data sources and notes
```


### Exercise 1: Prepare Data for Dashboard
**Task:** Save your processed data to files that the dashboard will load.
- Export cleaned plant data
- Export merged plant + LitPop exposure data
- Export company-level aggregations
- Save as CSV or Parquet for efficient loading


In [57]:
# Save processed datasets
#
# Written to data/processed/ in both Parquet (fast, typed -- used by app.py)
# and CSV (portable, inspectable). These files are committed so the Streamlit
# app runs on Streamlit Cloud without the raw source data.

PLANT_EXPORT_COLS = [
    "GEM plant ID", "Plant name (English)", "Owner", "Parent (English)",
    "Country/area", "Region", "Latitude", "Longitude",
    "capacity_ttpa", "n_operating_units", "Plant age (years)",
    "Main production equipment",
]

# 1. Cleaned global plant data (all 1,293 plants, Parts 1-3).
plants_export = plants[[c for c in PLANT_EXPORT_COLS if c in plants.columns]].copy()

# 2. Plants merged with LitPop exposure (CHN/IND/JPN subset, Part 4).
plants_litpop_export = plants_merged.copy()

# 3. Company-level aggregation (Part 5).
companies_export = companies.copy()

exports = {
    "plants_clean": plants_export,
    "plants_litpop": plants_litpop_export,
    "companies": companies_export,
}

for name, frame in exports.items():
    parquet_path = EXPORT_DIR / f"{name}.parquet"
    csv_path = EXPORT_DIR / f"{name}.csv"
    frame.to_parquet(parquet_path, index=False)
    frame.to_csv(csv_path, index=False)
    print(f"{name:<16} {frame.shape[0]:>6,} rows x {frame.shape[1]:>2} cols  ->  "
          f"{parquet_path.name} ({parquet_path.stat().st_size / 1024:>7,.1f} KB), "
          f"{csv_path.name} ({csv_path.stat().st_size / 1024:>7,.1f} KB)")

# Round-trip check: the dashboard must read back exactly what was written.
for name, frame in exports.items():
    reloaded = pd.read_parquet(EXPORT_DIR / f"{name}.parquet")
    assert reloaded.shape == frame.shape, f"{name}: shape changed on round-trip"
print("\nRound-trip verified: all exports reload with identical shapes.")

# Small metadata sidecar so the dashboard can state its own provenance.
metadata = pd.DataFrame([
    {"key": "source_plants", "value": PLANT_XLSX.name},
    {"key": "source_litpop", "value": ", ".join(p.name for p in litpop_files)},
    {"key": "litpop_ref_year", "value": str(metas[next(iter(metas))]["ref_year"])},
    {"key": "litpop_fin_mode", "value": "pc (produced capital)"},
    {"key": "litpop_value_unit", "value": "USD"},
    {"key": "litpop_resolution", "value": "300 arcsec (~10 km)"},
    {"key": "litpop_coverage", "value": ", ".join(covered_countries)},
    {"key": "neighbourhood_radius_km", "value": str(int(NEIGHBOURHOOD_RADIUS_KM))},
    {"key": "capacity_definition", "value": "sum of operating crude steel units (ttpa)"},
    {"key": "n_plants_global", "value": str(len(plants))},
    {"key": "n_plants_litpop", "value": str(len(plants_merged))},
    {"key": "n_companies", "value": str(len(companies))},
])
metadata.to_csv(EXPORT_DIR / "metadata.csv", index=False)
print(f"Wrote {(EXPORT_DIR / 'metadata.csv').name}")
display(metadata)

_sizes_kb = {n: (EXPORT_DIR / f"{n}.parquet").stat().st_size / 1024 for n in exports}
_rows_md = " · ".join(f"**{n}** ({f.shape[0]:,} rows)" for n, f in exports.items())
display(Markdown(f"""
> ### 💾 Processed data exported
> - {_rows_md} written to `data/processed/` as Parquet + CSV.
> - Total Parquet size: **{sum(_sizes_kb.values()):,.0f} KB** -- small enough for Streamlit Cloud.
> - Round-trip verified: every export reloads with an identical shape.
""".strip()))

plants_clean      1,293 rows x 12 cols  ->  plants_clean.parquet (  123.7 KB), plants_clean.csv (  254.3 KB)
plants_litpop       613 rows x 19 cols  ->  plants_litpop.parquet (   89.3 KB), plants_litpop.csv (  188.4 KB)
companies           535 rows x 22 cols  ->  companies.parquet (   87.5 KB), companies.csv (  140.0 KB)

Round-trip verified: all exports reload with identical shapes.
Wrote metadata.csv


,key,value
0,source_plants,Plant-level_data_Global_Iron_and_Steel_Tracker...
1,source_litpop,"LitPop_pc_300_arcsec_CHN_v1.hdf5, LitPop_pc_30..."
2,litpop_ref_year,2018
3,litpop_fin_mode,pc (produced capital)
4,litpop_value_unit,USD
5,litpop_resolution,300 arcsec (~10 km)
6,litpop_coverage,"China, India, Japan"
7,neighbourhood_radius_km,25
8,capacity_definition,sum of operating crude steel units (ttpa)
9,n_plants_global,1293


> ### 💾 Processed data exported
> - **plants_clean** (1,293 rows) · **plants_litpop** (613 rows) · **companies** (535 rows) written to `data/processed/` as Parquet + CSV.
> - Total Parquet size: **301 KB** -- small enough for Streamlit Cloud.
> - Round-trip verified: every export reloads with an identical shape.

### Exercise 2: Display relevant information from your exploratory analysis into the dashboard

In [58]:
# This cell is for notes/observations about your dashboard
#
# Run the dashboard with:  uv run streamlit run app.py

print("""
DASHBOARD NOTES
===============

What works well
---------------
* Data is precomputed. app.py reads three small Parquet files from
  data/processed/ instead of parsing a 736 KB workbook and 17 MB of HDF5 on
  every interaction. Cold start is well under a second.
* @st.cache_data on the loaders means filter changes never re-read from disk.
* The filters compose: region -> country -> company -> capacity range, each
  narrowing the next, with KPIs, map and table all driven by one filtered frame.
* Exposure colour is on a log10 scale. On a linear scale Shanghai and Tokyo
  saturate the palette and every other plant renders the same colour.
* Open-source basemaps (map_style='open-street-map') need no Mapbox token, so
  the app deploys to Streamlit Cloud with no secrets configured.

What could be improved
----------------------
* LitPop coverage is the real limitation: CHN / IND / JPN only. The global tab
  shows all 1,293 plants, but every exposure view is restricted to the 613
  plants in those three countries. Adding more LitPop country files would
  widen this without any code change -- the loader globs the directory.
* 'Owner' is a free-text field. Subsidiaries appear as separate companies
  (e.g. several 'Angang ...' entities), so company-level totals are
  fragmented. Grouping on 'Parent (English)' or the GEM entity ID would give a
  truer corporate footprint.
* Exposure is a 2018 snapshot. Pairing it with plant start dates would allow a
  built-before/built-after comparison rather than a static cross-section.
* The scatter map draws one marker per plant with no clustering; at country
  zoom the dense Chinese coastal cluster overplots.

Performance
-----------
* Dataset is small (1,293 plants / 613 merged / ~535 companies) so everything
  is instant. The expensive step is the spatial join, and it is done here in
  the notebook, not in the app.
* The BallTree join over 182,591 LitPop cells runs in well under a second.
  A brute-force pairwise distance matrix would be 613 x 182,591 -- the tree
  avoids materialising that.
* If LitPop were extended to all 91 countries, the grid would grow into the
  millions of cells. The per-country tree loop already keeps that tractable,
  since each tree only holds one country's cells.
""")


DASHBOARD NOTES

What works well
---------------
* Data is precomputed. app.py reads three small Parquet files from
  data/processed/ instead of parsing a 736 KB workbook and 17 MB of HDF5 on
  every interaction. Cold start is well under a second.
* @st.cache_data on the loaders means filter changes never re-read from disk.
* The filters compose: region -> country -> company -> capacity range, each
  narrowing the next, with KPIs, map and table all driven by one filtered frame.
* Exposure colour is on a log10 scale. On a linear scale Shanghai and Tokyo
  saturate the palette and every other plant renders the same colour.
* Open-source basemaps (map_style='open-street-map') need no Mapbox token, so
  the app deploys to Streamlit Cloud with no secrets configured.

What could be improved
----------------------
* LitPop coverage is the real limitation: CHN / IND / JPN only. The global tab
  shows all 1,293 plants, but every exposure view is restricted to the 613
  plants in those three co

---
## Bonus (optional): Deploy to Streamlit Cloud

Deploy your dashboard so it runs in the browser without local setup.

**Task:**
1. Make sure `app.py` and any required data files are in your GitHub repo
2. Go to [https://share.streamlit.io](https://share.streamlit.io) (Streamlit Community Cloud)
3. Sign in with GitHub, select your repo, and deploy `app.py`
4. Copy the public app URL

**Submit:** paste the Streamlit Cloud link in the **Submission info** section at the top **and** include it in your group's submission email (with the GitHub repo URL).


---
## Lab Summary and Key Takeaways

**What you learned:**
- How to perform EDA on geospatial datasets
- Creating interactive maps with Plotly for geospatial data
- Merging LitPop exposure / population data with assets based on geographic proximity
- Aggregating geospatial data at different levels (asset vs. company)
- Building interactive dashboards with Streamlit

**Next Steps:**
- Explore other geospatial libraries (GeoPandas, Folium, Kepler.gl)
- Learn about coordinate reference systems (CRS) and projections
- Practice with other datasets (buildings, utilities, transportation)
- (Bonus) Deploy your dashboard to Streamlit Cloud and share the link in your submission email
